In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import roc_auc_score
from tensorflow.keras.layers import Dense, Activation, BatchNormalization, LSTM, Masking, Input, GRU, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l1
from tensorflow.keras import regularizers
from sklearn.utils import shuffle
from qkeras import *
import qkeras
from tensorflow.keras.models import load_model
from qkeras.utils import model_quantize
from qkeras.utils import model_save_quantized_weights

import hls4ml

import os

import helper_functions

2026-04-30 00:15:57.955322: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-30 00:15:58.516790: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
def lstmmodel(max_len, n_var, rec_units, ndense=[10], l1_reg=0,
              l2_reg=0, rec_act='sigmoid', extra_lab='none', rec_kernel_init='VarianceScaling',
             dense_kernel_init='lecun_uniform', domask=False):
    
    hidden = x_in = Input(shape=(max_len, n_var,))

    hidden = LSTM(units=rec_units,
                  recurrent_activation = rec_act,
                  kernel_initializer = rec_kernel_init, 
                  name = 'lstm1')(hidden)

    hidden = Dense(50, kernel_initializer=dense_kernel_init, name='dense_0' )(hidden)
    hidden = Activation('relu', name = 'relu_0')(hidden)

    hidden = Dense(10, kernel_initializer=dense_kernel_init, name='dense_1' )(hidden)
    hidden = Activation('relu', name = 'relu_1')(hidden)

    hidden = Dense(3, kernel_initializer=dense_kernel_init, name='dense_2' )(hidden)
    hidden = Activation('softmax', name='output_softmax')(hidden)

    model = Model(inputs=x_in, outputs=hidden)
    
    return model

In [4]:
## lstm model
l1_reg = 0
l2_reg = 0

lstm_model = lstmmodel(15, 6, 120, [50, 10], l1_reg=l1_reg, l2_reg=l2_reg)
lstm_model.compile(optimizer='adam', loss=['categorical_crossentropy'], metrics=['accuracy'])

2026-04-30 00:16:03.030056: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
lstm_model.load_weights("./lstm_test2/lstm_weights.h5", skip_mismatch=True, by_name=True) 

print("lstm model")
lstm_model.summary()

#loading weights by layer to confirm it actually work
for layer in lstm_model.layers:
    #print(layer.name, layer.input_shape)
    
    weights = layer.get_weights()
    os.makedirs(f'./weights/weights_{layer.name}_weights_biases_pkgs', exist_ok=True)

    # getting types
    print(type(weights))
    print(type(layer.get_weights()))
    print(layer.name, layer.get_config())
    print(layer.name, layer.get_weights())
    
    for weight_list in layer.get_weights():
        for indv_weight in weight_list:
            print(str(indv_weight))
    
# below this uncomment out when want to generate weights
    with open(f'./weights/weights_{layer.name}_weights_biases_pkgs/{layer.name}_weights.txt', 'w') as f_weight:
        with open(f'./weights/weights_{layer.name}_weights_biases_pkgs/{layer.name}_biases.txt', 'w') as f_bias:
            for weight_list in layer.get_weights():
                for weight_set in weight_list:
                                    #print(type(weight_set))
                    if isinstance(weight_set, np.float32):
                        f_bias.write((str(weight_set))+"\n")
                    else:                    
                        for weight_indv in weight_set:
                                #print(type(weight_indv))
                            f_weight.write((str(weight_indv))+"\n")
                    #if isinstance(weight_set, 'numpy.float32'):
                    #    for weight_indv in weight_set:
                    #        f.write("".join(str(weight_indv))+"\n")
                    #else
                    #    f.write("".join(str(weight_indv))+"\n")
                #f.write(".\n")
            #f.write("\n".join(np.array(weight).to_list()))

# this helper function doesn't work, I am guessing because the size/ dimensions of my weights input were different than caleb's
#helper_functions.dump_weights(nogru_model)
        

lstm model
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 15, 6)]           0         
                                                                 
 lstm1 (LSTM)                (None, 120)               60960     
                                                                 
 dense_0 (Dense)             (None, 50)                6050      
                                                                 
 relu_0 (Activation)         (None, 50)                0         
                                                                 
 dense_1 (Dense)             (None, 10)                510       
                                                                 
 relu_1 (Activation)         (None, 10)                0         
                                                                 
 dense_2 (Dense)             (None, 3)            

<class 'list'>
<class 'list'>
input_1 {'batch_input_shape': (None, 15, 6), 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_1'}
input_1 []
<class 'list'>
<class 'list'>
lstm1 {'name': 'lstm1', 'trainable': True, 'dtype': 'float32', 'return_sequences': False, 'return_state': False, 'go_backwards': False, 'stateful': False, 'unroll': False, 'time_major': False, 'units': 120, 'activation': 'tanh', 'recurrent_activation': 'sigmoid', 'use_bias': True, 'kernel_initializer': {'class_name': 'VarianceScaling', 'config': {'scale': 1.0, 'mode': 'fan_in', 'distribution': 'truncated_normal', 'seed': None}}, 'recurrent_initializer': {'class_name': 'Orthogonal', 'config': {'gain': 1.0, 'seed': None}}, 'bias_initializer': {'class_name': 'Zeros', 'config': {}}, 'unit_forget_bias': True, 'kernel_regularizer': None, 'recurrent_regularizer': None, 'bias_regularizer': None, 'activity_regularizer': None, 'kernel_constraint': None, 'recurrent_constraint': None, 'bias_constraint': None, 'd

In [6]:
x_test_toptag = np.load('/home/quin/HLS4ML_VS_MANUAL/documents/Benchmarks/Btagging/qkeras/traindata_toptag/x_test.npy')
x_test_lstm_inport = np.ascontiguousarray(x_test_toptag)
#print(x_test_gru_inport)

y_lstm = np.resize(x_test_lstm_inport, (x_test_lstm_inport.shape[0], 15, 6)) #x_test_gru_inport.shape[0], 

predict_lstm = lstm_model.predict(y_lstm)
#print(predict_lstm)

# making 0's
y_test_gru_zeros = np.zeros((x_test_lstm_inport.shape[0], 15, 6))
#print(y_test_gru_zeros)

# zeroed input data
predict_lstm_model_zeros = lstm_model.predict(y_test_gru_zeros)
#print(predict_lstm_model_zeros)

## writing the training data files
os.makedirs(f'./acc/testingData_lstm', exist_ok=True)
acc = (16, 6)
#helper_functions.gen_test(acc, y_test_gru_zeros, "zeroesData")
helper_functions.gen_test(acc, y_lstm, "acc/testingData_lstm/", "toptaggingData")

## writing the predicted outputs files, uncomment out if need to make outputs again 
os.makedirs(f'./acc/lstm', exist_ok=True)

with open(f'./acc/lstm/keras_zeros.txt', 'w') as f: #_noSoft
     for y_nogru_zeroes_layers in predict_lstm_model_zeros:
        pos = -1
        for y_nogru_zeroes_individual in y_nogru_zeroes_layers:
            pos += 1
            if pos == y_nogru_zeroes_layers.size-1:
                f.write(str(y_nogru_zeroes_individual))
            else:
                f.write(str(y_nogru_zeroes_individual)+",")
        f.write("\n")
            
with open(f'./acc/lstm/keras_predict.txt', 'w') as f: #_noSoft
    for y_nogru_layers in predict_lstm:
        pos = -1
        for y_nogru_individual in y_nogru_layers:
            pos += 1
            if pos == y_nogru_layers.size-1:
                f.write(str(y_nogru_individual))
            else:
                f.write(str(y_nogru_individual)+",")
        f.write("\n")
    #f.write(str(y_nogru))
#np.savetxt(f"./acc/keras_predict.txt", y_nogru), fmt='%.4f')


#y_nogru_model_pred = np.argmax(y_nogru, axis=1)
#y_nogru_handmade = np.argmax(, axis=1)

#print("no gru model Acurracy: {}".format(accuracy_score(np.argmax(y_nogru, axis=1) np.argmax(, axis=1))))

#from sklearn.metrics import classification_report, confusion_matrix

#print (classification_report(y_nogru_model_pred, y_nogru_handmade))



624/624 [==============================] - 2s 3ms/step


In [7]:
## Gen weights
acc = (16, 6)
#filepath = "lstm/weights/"
filepath = "weights"
helper_functions.gen_weight(acc, filepath, lstm_model)

# I would need to keep track of each layer for the gen_weight_txt and change it in helper function,
# the order ends up being different and errors and I can fix it but it creates more problems
#weights_file = "./weights/weights_dense_0_weights_biases_pkgs/dense_0_weights.txt"
#biases_file = "./weights/weights_dense_0_weights_biases_pkgs/dense_0_biases.txt"
#helper_functions.gen_weight_txt(acc)

#error=f"No_gru Predict Script Completed"
#os.system(f'printf "Complete converting data to binary" | mail -s "{error}" ljdono@uw.edu')

2 1


**HLS**

In [ ]:
config = hls4ml.utils.config_from_keras_model(lstm_model, granularity='model')
#config = hls4ml.utils.config_from_keras_model(model, granularity='model')
                                # backend='Vivado' #granularity = 'name' #granularity = 'model' 
                                # default_precision='ap_fixed<16,6>' # default_reuse_factor=2
config['Model']['ReuseFactor'] = 10

config['Model']['Strategy'] = 'resource'  #stable
config['Model']['Precision'] = 'ap_fixed<6,3>' 

output = 'LSTM_6_3_reuse10_bigboard_stream'

#creating hls model using 'Vivado'
backendSynth = 'Vivado' #vitis'

hls_model = hls4ml.converters.convert_from_keras_model(
    #model, 
    lstm_model,
    hls_config=config, 
    backend=backendSynth, 
    output_dir=(f'models/lstm_floating_point/{output}/hls4ml_prj'), #fifoop_
    part='xcvu13p-fhga2104-3-e',
    io_type='io_stream'
) 

In [10]:
hls_model.compile()

In [ ]:
x_test_toptag = np.load('/home/quin/HLS4ML_VS_MANUAL/documents/Benchmarks/Btagging/qkeras/traindata_toptag/x_test.npy')
x_test_lstm_inport = np.ascontiguousarray(x_test_toptag)

y_lstm = np.resize(x_test_lstm_inport, (x_test_lstm_inport.shape[0], 15, 6)) #x_test_gru_inport.shape[0], 

predict_lstm_hls = hls_model.predict(y_lstm)

# making 0's
y_test_gru_zeros = np.zeros((x_test_lstm_inport.shape[0], 15, 6))

# zeroed input data
predict_lstm_hls_zeros = hls_model.predict(y_test_gru_zeros)

## writing the training data files
os.makedirs(f'./acc/testingData_lstm', exist_ok=True)
acc = (16, 6)
#helper_functions.gen_test(acc, y_test_gru_zeros, "zeroesData")
#helper_functions.gen_test(acc, y_lstm, "acc/testingData_lstm/", "toptaggingData")

## writing the predicted outputs files, uncomment out if need to make outputs again 
os.makedirs(f'./acc/lstm', exist_ok=True)

with open(f'./acc/lstm/hls_keras_6_3_zeros.txt', 'w') as f: #_noSoft
     for y_nogru_zeroes_layers in predict_lstm_hls_zeros:
        pos = -1
        for y_nogru_zeroes_individual in y_nogru_zeroes_layers:
            pos += 1
            if pos == y_nogru_zeroes_layers.size-1:
                f.write(str(y_nogru_zeroes_individual))
            else:
                f.write(str(y_nogru_zeroes_individual)+",")
        f.write("\n")
            
with open(f'./acc/lstm/hls_keras_6_3_predict.txt', 'w') as f: #_noSoft
    for y_nogru_layers in predict_lstm_hls:
        pos = -1
        for y_nogru_individual in y_nogru_layers:
            pos += 1
            if pos == y_nogru_layers.size-1:
                f.write(str(y_nogru_individual))
            else:
                f.write(str(y_nogru_individual)+",")
        f.write("\n")
    #f.write(str(y_nogru))
#np.savetxt(f"./acc/keras_predict.txt", y_nogru), fmt='%.4f')


#y_nogru_model_pred = np.argmax(y_nogru, axis=1)
#y_nogru_handmade = np.argmax(, axis=1)

#print("no gru model Acurracy: {}".format(accuracy_score(np.argmax(y_nogru, axis=1) np.argmax(, axis=1))))

#from sklearn.metrics import classification_report, confusion_matrix

#print (classification_report(y_nogru_model_pred, y_nogru_handmade))

In [ ]:
# build the model using HLS
hls_model.build(csim=False) #

acc = (6,3)
name = "btag"

# remember to change hls_script to pull from the right file
#os.system(f"vivado -mode batch -source hls_script.tcl")
os.system(f'vivado -mode batch -source hls_script.tcl -tclargs {acc[0]}_{acc[1]}_{name}')

file_path_report = (f'/home/quin/HLS4ML_VS_MANUAL/documents/Benchmarks/Btagging/qkeras/reports/lstm_hls/*_util.rpt')

error=f"HLS4ML Script Completed"
if os.path.exists(file_path_report):
    os.system(f'printf "Completed building HLS model using {backendSynth}, no errors" | mail -s "{error}" ljdono@uw.edu') 
else:
    error=f"HLS4ML Script Completed"
    os.system(f'printf "Couldnt complete building HLS model using {backendSynth}, errored out" | mail -s "{error}" ljdono@uw.edu') 
